In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import cdist
from xgboost import XGBRegressor

# ۱. خواندن فایل اکسل اصلی
file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
output_filename = r'outputs\G11\dsas_g11_bearings_vibration_temp_deviation_monitoring\deviation_monitoring\dsas_g11_bearings_vibration_temp_deviation_monitoring_output1.xlsx'
try:
    df_raw = pd.read_excel(file_path)
    print("فایل اصلی با موفقیت خوانده شد.")
except Exception as e:
    print(f"خطا در خواندن فایل: {e}")
    exit()

# لیست تمام فیچرهای عددی
all_features = [
    'AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361', 
    'AssetID_9368', 'AssetID_9369', 'AssetID_9370', 'AssetID_12208',
    'AssetID_9343', 'AssetID_9344', 'AssetID_9408'
]

# لیست سنسورهایی که تارگت هستند
target_sensors = [
    'AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361'
]

# ۲. مرحله پیش‌پردازش و حذف ۱۰ درصد داده‌های پرت با DBSCAN
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_raw[all_features])

dbscan = DBSCAN(eps=0.5, min_samples=5)
labels = dbscan.fit_predict(scaled_data)

cluster_centers = {}
for cluster_id in set(labels):
    if cluster_id != -1:
        cluster_centers[cluster_id] = scaled_data[labels == cluster_id].mean(axis=0)

def calculate_distance(row_index):
    label = labels[row_index]
    point = scaled_data[row_index].reshape(1, -1)
    if label != -1:
        center = cluster_centers[label].reshape(1, -1)
        return cdist(point, center, metric='euclidean')[0][0]
    else:
        if not cluster_centers: return 0.0
        all_centers = np.array(list(cluster_centers.values()))
        return np.min(cdist(point, all_centers, metric='euclidean'))

df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]

# حذف ۱۰ درصد داده‌های با بیشترین فاصله
df_sorted = df_raw.sort_values(by='distance', ascending=False)
num_to_remove = int(len(df_sorted)*0.10)
df_cleaned = df_sorted.iloc[num_to_remove:].copy()

# سورت زمانی و تفکیک Test/Train
df_cleaned['date'] = pd.to_datetime(df_cleaned['date'])
df_cleaned = df_cleaned.sort_values(by='date')

split_idx = int(len(df_cleaned) * 0.80)
train_df = df_cleaned.iloc[:split_idx].copy()
test_df = df_cleaned.iloc[split_idx:].copy()

# ۳. شروع حلقه پیش‌بینی برای هر سنسور و اضافه کردن به یک دیتافریم واحد
print("شروع فرآیند یادگیری و پیش‌بینی برای تمام سنسورها...")

for target in target_sensors:
    # تعیین فیچرها (همه بجز تارگت فعلی)
    current_features = [f for f in all_features if f != target]

    X_train, y_train = train_df[current_features], train_df[target]
    X_test = test_df[current_features]

    # آموزش مدل XGBoost
    model = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)

    model.fit(X_train, y_train)

    # ثبت پیش‌بینی در یک ستون جدید با نام اختصاصی
    column_name = f'predicted_{target.split("_")[1]}'
    test_df[column_name] = model.predict(X_test)
    print(f"پیش‌بینی {target} با موفقیت انجام شد.")

# ۴. ذخیره نهایی در یک فایل اکسل واحد
# مرتب‌سازی ستون‌ها برای خوانایی بهتر (ابتدا تاریخ، سپس سنسورهای واقعی و در انتها پیش‌بینی‌ها)
predicted_cols = [f'predicted_{t.split("_")[1]}' for t in target_sensors]
final_columns_order = ['date'] + all_features + predicted_cols

test_df[final_columns_order].to_excel(output_filename, index=False)

print(f"\nعملیات با موفقیت پایان یافت. فایل جامع '{output_filename}' ایجاد شد.")
print(f"این فایل شامل {len(test_df)} ردیف و ستون‌های پیش‌بینی مجزا برای هر سنسور است.")

فایل اصلی با موفقیت خوانده شد.
شروع فرآیند یادگیری و پیش‌بینی برای تمام سنسورها...
پیش‌بینی AssetID_9358 با موفقیت انجام شد.
پیش‌بینی AssetID_9359 با موفقیت انجام شد.
پیش‌بینی AssetID_9360 با موفقیت انجام شد.
پیش‌بینی AssetID_9361 با موفقیت انجام شد.

عملیات با موفقیت پایان یافت. فایل جامع 'outputs\G11\dsas_g11_bearings_vibration_temp_deviation_monitoring\deviation_monitoring\dsas_g11_bearings_vibration_temp_deviation_monitoring_output1.xlsx' ایجاد شد.
این فایل شامل 1059 ردیف و ستون‌های پیش‌بینی مجزا برای هر سنسور است.
